# DataVisualisation-TP2025 — TP7

[Teacher assignment](https://github.com/ahmed3991/DataVisualisation-TP2025/blob/main/TP7/TP7.md)

Run the cells in order. Private credentials are read from an ignored .env file or hidden input. Read EXPLICATION_AR.md for the Arabic explanation.

Run TP6 first. This notebook needs your Supabase connection AND a running Metabase server. On this Windows computer, use the local Metabase instance at http://localhost:8000. A local Windows server is not automatically reachable from Colab; running this notebook locally is the simplest method.

## Install required libraries if needed

In [1]:
import importlib.util
import subprocess
import sys
packages = {'pandas': 'pandas', 'requests': 'requests', 'psycopg2': 'psycopg2-binary', 'dotenv': 'python-dotenv'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])


## Companion files

In [2]:
from pathlib import Path
# Recreate the companion source/configuration files; no passwords are embedded.
support_files = {'connection.py': '"""Read a private PostgreSQL connection for both dlt and dbt."""\nimport os\nfrom getpass import getpass\nfrom pathlib import Path\nfrom urllib.parse import quote, urlparse, unquote, parse_qs\nfrom dotenv import load_dotenv\n\nROOT = Path(__file__).resolve().parent\n\ndef configure_connection():\n    load_dotenv(ROOT / ".env")\n    url = os.environ.get("DATABASE_URL")\n    if not url:\n        password = os.environ.get("DATABASE_PASSWORD") or getpass("Supabase database password (hidden): ")\n        host = os.environ.get("SUPABASE_HOST") or input("Supabase session pooler host: ").strip()\n        user = os.environ.get("SUPABASE_USER") or input("Supabase database user: ").strip()\n        port = os.environ.get("SUPABASE_PORT", "5432")\n        database = os.environ.get("SUPABASE_DATABASE", "postgres")\n        url = f"postgresql://{quote(user, safe=\'\')}:{quote(password, safe=\'\')}@{host}:{port}/{database}?sslmode=require"\n        os.environ["DATABASE_URL"] = url\n    parsed = urlparse(url)\n    if parsed.scheme not in {"postgres", "postgresql"}:\n        raise ValueError("A PostgreSQL connection URL is required.")\n    os.environ.update({"PGHOST": parsed.hostname or "localhost", "PGPORT": str(parsed.port or 5432),\n        "PGUSER": unquote(parsed.username or ""), "PGPASSWORD": unquote(parsed.password or ""),\n        "PGDATABASE": unquote(parsed.path.lstrip("/")),\n        "PGSSLMODE": parse_qs(parsed.query).get("sslmode", ["require"])[0]})\n    return url\n', 'metabase_client.py': '"""Small client for local Metabase, with private credentials."""\nfrom getpass import getpass\nimport json\nimport os\nfrom pathlib import Path\nimport requests\nfrom dotenv import load_dotenv\n\nROOT = Path(__file__).resolve().parent\n\nclass MetabaseClient:\n    def __init__(self):\n        load_dotenv(ROOT / ".env")\n        self.url = os.environ.get("METABASE_URL", "http://localhost:8000").rstrip("/")\n        self.session = requests.Session()\n        credentials_file = ROOT / ".secrets" / "metabase.json"\n        local = json.loads(credentials_file.read_text()) if credentials_file.exists() else {}\n        email = os.environ.get("METABASE_EMAIL") or local.get("email") or input("Metabase login email: ").strip()\n        password = os.environ.get("METABASE_PASSWORD") or local.get("password") or getpass("Metabase password (hidden): ")\n        result = self.request("POST", "/api/session", {"username": email, "password": password})\n        self.session.headers["X-Metabase-Session"] = result["id"]\n\n    def request(self, method, path, data=None):\n        response = self.session.request(method, self.url + path, json=data, timeout=120)\n        response.raise_for_status()\n        return response.json() if response.content else None\n', 'create_dashboard.py': '"""Create TP7\'s Metabase dashboard using the real TP6 Supabase warehouse."""\nimport json\nimport os\nfrom pathlib import Path\nimport time\nimport uuid\nfrom connection import configure_connection\nfrom metabase_client import MetabaseClient\n\nROOT = Path(__file__).resolve().parent\nFILTERS = {\n    "date_range": ("Order date range", "date/range", "fct_order_details", "order_date"),\n    "category": ("Product category", "string/=", "dim_categories", "category_name"),\n    "country": ("Customer country", "string/=", "dim_customers", "country"),\n    "employee": ("Employee", "string/=", "dim_employees", "employee_full_name"),\n    "product": ("Product name", "string/=", "dim_products", "product_name"),\n}\nCARDS = [\n    ("01_total_revenue", "Total Revenue", "scalar", None, "total_revenue"),\n    ("02_total_orders", "Total Orders", "scalar", None, "total_orders"),\n    ("03_average_order_value", "Average Order Value", "scalar", None, "average_order_value"),\n    ("04_total_quantity", "Total Quantity Sold", "scalar", None, "total_quantity"),\n    ("05_sales_trend", "Sales Trend by Month", "line", "month", "revenue"),\n    ("06_sales_by_category", "Sales by Product Category", "bar", "category", "revenue"),\n    ("07_top_products", "Top 10 Products by Revenue", "bar", "product", "revenue"),\n    ("08_sales_by_country", "Sales by Customer Country", "bar", "country", "revenue"),\n    ("09_employee_performance", "Employee Performance", "bar", "employee", "revenue"),\n    ("10_shipping_performance", "Average Shipping Days by Employee", "bar", "employee", "average_shipping_days"),\n]\n\ndef create_dashboard():\n    configure_connection()\n    client = MetabaseClient()\n    state_path = ROOT / ".secrets" / "dashboard_state.json"\n    state = json.loads(state_path.read_text()) if state_path.exists() else {}\n    state_path.parent.mkdir(exist_ok=True)\n    if not state.get("database_id"):\n        database = client.request("POST", "/api/database", {\n            "name": "Northwind Supabase", "engine": "postgres", "is_full_sync": False,\n            "details": {"host": os.environ["PGHOST"], "port": int(os.environ["PGPORT"]),\n                "dbname": os.environ["PGDATABASE"], "user": os.environ["PGUSER"],\n                "password": os.environ["PGPASSWORD"], "ssl": True,\n                "ssl-mode": "require", "schema-filters-type": "inclusion",\n                "schema-filters-patterns": "northwind_warehouse"}})\n        state["database_id"] = database["id"]\n        state_path.write_text(json.dumps(state, indent=2), encoding="utf-8")\n    database_id = state["database_id"]\n    client.request("POST", f"/api/database/{database_id}/sync_schema", {})\n    for attempt in range(40):\n        metadata = client.request("GET", f"/api/database/{database_id}/metadata")\n        field_ids = {(table["name"], field["name"]): field["id"]\n                     for table in metadata.get("tables", []) if table.get("schema") == "northwind_warehouse"\n                     for field in table.get("fields", [])}\n        if all((table, field) in field_ids for _, _, table, field in FILTERS.values()):\n            break\n        time.sleep(2)\n    else:\n        raise RuntimeError("TP6 warehouse fields have not appeared in Metabase; run TP6 first and retry.")\n    parameters = [{"id": name, "name": label, "slug": name, "type": kind,\n                   "isMultiSelect": kind != "date/range", "required": False,\n                   **({"values_query_type": "list"} if kind != "date/range" else {})}\n                  for name, (label, kind, _, _) in FILTERS.items()]\n    for name, (_, kind, table, field) in FILTERS.items():\n        if kind != "date/range":\n            identifier = field_ids[(table, field)]\n            client.request("PUT", f"/api/field/{identifier}", {"has_field_values": "list", "semantic_type": "type/Category"})\n            client.request("POST", f"/api/field/{identifier}/rescan_values", {})\n    if not state.get("dashboard_id"):\n        dashboard = client.request("POST", "/api/dashboard", {\n            "name": "Northwind Order Details — TP7", "parameters": parameters,\n            "description": "Northwind sales, customers, products, employees and shipping. Revenue uses the source\'s monetary unit."})\n        state["dashboard_id"] = dashboard["id"]\n        state_path.write_text(json.dumps(state, indent=2), encoding="utf-8")\n    dashboard_id = state["dashboard_id"]\n    client.request("PUT", f"/api/dashboard/{dashboard_id}", {"parameters": parameters, "width": "full"})\n    cards = []\n    state.setdefault("card_ids", {})\n    for index, (filename, label, display, dimension, metric) in enumerate(CARDS):\n        tags = {name: {"id": str(uuid.uuid5(uuid.NAMESPACE_URL, filename + name)), "name": name,\n                       "display-name": title, "type": "dimension",\n                       "dimension": ["field", field_ids[(table, field)], None],\n                       "widget-type": kind, "required": False, "default": None}\n                for name, (title, kind, table, field) in FILTERS.items()}\n        query = {"database": database_id, "type": "native", "native": {\n            "query": (ROOT / "queries" / f"{filename}.sql").read_text(encoding="utf-8"), "template-tags": tags}}\n        settings = {"scalar.decimals": 2 if metric in {"total_revenue", "average_order_value"} else 0} if display == "scalar" else {\n            "graph.dimensions": [dimension], "graph.metrics": [metric], "graph.show_values": False}\n        card_data = {"name": label, "display": display, "dataset_query": query,\n                     "visualization_settings": settings, "parameters": [\n                         {"id": name, "type": kind, "name": title, "slug": name,\n                          "target": ["dimension", ["template-tag", name]]}\n                         for name, (title, kind, _, _) in FILTERS.items()]}\n        existing = state["card_ids"].get(filename)\n        card = client.request("PUT" if existing else "POST", f"/api/card/{existing}" if existing else "/api/card", card_data)\n        card_id = card["id"]\n        state["card_ids"][filename] = card_id\n        state_path.write_text(json.dumps(state, indent=2), encoding="utf-8")\n        if index < 4:\n            row, col, width, height = 0, index * 6, 6, 4\n        else:\n            row, col, width, height = 4 + ((index - 4) // 2) * 7, ((index - 4) % 2) * 12, 12, 7\n        cards.append({"id": -(index + 1), "card_id": card_id, "row": row, "col": col,\n                      "size_x": width, "size_y": height, "series": [],\n                      "parameter_mappings": [{"parameter_id": name, "card_id": card_id,\n                           "target": ["dimension", ["template-tag", name]]} for name in FILTERS],\n                      "visualization_settings": {}})\n    current = client.request("GET", f"/api/dashboard/{dashboard_id}")\n    existing_slots = {slot["card_id"]: slot["id"] for slot in current.get("dashcards", [])}\n    for card in cards:\n        card["id"] = existing_slots.get(card["card_id"], card["id"])\n    client.request("PUT", f"/api/dashboard/{dashboard_id}/cards", {"cards": cards})\n    print("Created four KPIs, six charts and five connected filters.")\n    print("Dashboard:", f"{client.url}/dashboard/{dashboard_id}")\n    return client, state\n\nif __name__ == "__main__":\n    create_dashboard()\n', 'queries/01_total_revenue.sql': 'with filtered as (\n    select fct_order_details.order_id, fct_order_details.order_date,\n           fct_order_details.extended_price, fct_order_details.quantity,\n           fct_order_details.shipping_delay_days,\n           dim_categories.category_name, dim_products.product_name,\n           dim_customers.country, dim_employees.employee_full_name,\n           dim_suppliers.company_name as supplier_name\n    from northwind_warehouse.fct_order_details\n    join northwind_warehouse.dim_customers using (customer_pk)\n    join northwind_warehouse.dim_employees using (employee_pk)\n    join northwind_warehouse.dim_products using (product_pk)\n    join northwind_warehouse.dim_categories on dim_products.category_id = dim_categories.category_id\n    join northwind_warehouse.dim_suppliers on dim_products.supplier_id = dim_suppliers.supplier_id\n    where 1 = 1\n      [[and {{date_range}}]]\n      [[and {{category}}]]\n      [[and {{country}}]]\n      [[and {{employee}}]]\n      [[and {{product}}]]\n)\nselect coalesce(sum(extended_price), 0) as total_revenue from filtered;\n', 'queries/02_total_orders.sql': 'with filtered as (\n    select fct_order_details.order_id, fct_order_details.order_date,\n           fct_order_details.extended_price, fct_order_details.quantity,\n           fct_order_details.shipping_delay_days,\n           dim_categories.category_name, dim_products.product_name,\n           dim_customers.country, dim_employees.employee_full_name,\n           dim_suppliers.company_name as supplier_name\n    from northwind_warehouse.fct_order_details\n    join northwind_warehouse.dim_customers using (customer_pk)\n    join northwind_warehouse.dim_employees using (employee_pk)\n    join northwind_warehouse.dim_products using (product_pk)\n    join northwind_warehouse.dim_categories on dim_products.category_id = dim_categories.category_id\n    join northwind_warehouse.dim_suppliers on dim_products.supplier_id = dim_suppliers.supplier_id\n    where 1 = 1\n      [[and {{date_range}}]]\n      [[and {{category}}]]\n      [[and {{country}}]]\n      [[and {{employee}}]]\n      [[and {{product}}]]\n)\nselect count(distinct order_id) as total_orders from filtered;\n', 'queries/03_average_order_value.sql': 'with filtered as (\n    select fct_order_details.order_id, fct_order_details.order_date,\n           fct_order_details.extended_price, fct_order_details.quantity,\n           fct_order_details.shipping_delay_days,\n           dim_categories.category_name, dim_products.product_name,\n           dim_customers.country, dim_employees.employee_full_name,\n           dim_suppliers.company_name as supplier_name\n    from northwind_warehouse.fct_order_details\n    join northwind_warehouse.dim_customers using (customer_pk)\n    join northwind_warehouse.dim_employees using (employee_pk)\n    join northwind_warehouse.dim_products using (product_pk)\n    join northwind_warehouse.dim_categories on dim_products.category_id = dim_categories.category_id\n    join northwind_warehouse.dim_suppliers on dim_products.supplier_id = dim_suppliers.supplier_id\n    where 1 = 1\n      [[and {{date_range}}]]\n      [[and {{category}}]]\n      [[and {{country}}]]\n      [[and {{employee}}]]\n      [[and {{product}}]]\n)\nselect coalesce(sum(extended_price) / nullif(count(distinct order_id), 0), 0) as average_order_value from filtered;\n', 'queries/04_total_quantity.sql': 'with filtered as (\n    select fct_order_details.order_id, fct_order_details.order_date,\n           fct_order_details.extended_price, fct_order_details.quantity,\n           fct_order_details.shipping_delay_days,\n           dim_categories.category_name, dim_products.product_name,\n           dim_customers.country, dim_employees.employee_full_name,\n           dim_suppliers.company_name as supplier_name\n    from northwind_warehouse.fct_order_details\n    join northwind_warehouse.dim_customers using (customer_pk)\n    join northwind_warehouse.dim_employees using (employee_pk)\n    join northwind_warehouse.dim_products using (product_pk)\n    join northwind_warehouse.dim_categories on dim_products.category_id = dim_categories.category_id\n    join northwind_warehouse.dim_suppliers on dim_products.supplier_id = dim_suppliers.supplier_id\n    where 1 = 1\n      [[and {{date_range}}]]\n      [[and {{category}}]]\n      [[and {{country}}]]\n      [[and {{employee}}]]\n      [[and {{product}}]]\n)\nselect coalesce(sum(quantity), 0) as total_quantity from filtered;\n', 'queries/05_sales_trend.sql': "with filtered as (\n    select fct_order_details.order_id, fct_order_details.order_date,\n           fct_order_details.extended_price, fct_order_details.quantity,\n           fct_order_details.shipping_delay_days,\n           dim_categories.category_name, dim_products.product_name,\n           dim_customers.country, dim_employees.employee_full_name,\n           dim_suppliers.company_name as supplier_name\n    from northwind_warehouse.fct_order_details\n    join northwind_warehouse.dim_customers using (customer_pk)\n    join northwind_warehouse.dim_employees using (employee_pk)\n    join northwind_warehouse.dim_products using (product_pk)\n    join northwind_warehouse.dim_categories on dim_products.category_id = dim_categories.category_id\n    join northwind_warehouse.dim_suppliers on dim_products.supplier_id = dim_suppliers.supplier_id\n    where 1 = 1\n      [[and {{date_range}}]]\n      [[and {{category}}]]\n      [[and {{country}}]]\n      [[and {{employee}}]]\n      [[and {{product}}]]\n)\nselect date_trunc('month', order_date)::date as month, sum(extended_price) as revenue from filtered group by 1 order by 1;\n", 'queries/06_sales_by_category.sql': 'with filtered as (\n    select fct_order_details.order_id, fct_order_details.order_date,\n           fct_order_details.extended_price, fct_order_details.quantity,\n           fct_order_details.shipping_delay_days,\n           dim_categories.category_name, dim_products.product_name,\n           dim_customers.country, dim_employees.employee_full_name,\n           dim_suppliers.company_name as supplier_name\n    from northwind_warehouse.fct_order_details\n    join northwind_warehouse.dim_customers using (customer_pk)\n    join northwind_warehouse.dim_employees using (employee_pk)\n    join northwind_warehouse.dim_products using (product_pk)\n    join northwind_warehouse.dim_categories on dim_products.category_id = dim_categories.category_id\n    join northwind_warehouse.dim_suppliers on dim_products.supplier_id = dim_suppliers.supplier_id\n    where 1 = 1\n      [[and {{date_range}}]]\n      [[and {{category}}]]\n      [[and {{country}}]]\n      [[and {{employee}}]]\n      [[and {{product}}]]\n)\nselect category_name as category, sum(extended_price) as revenue from filtered group by 1 order by 2 desc;\n', 'queries/07_top_products.sql': 'with filtered as (\n    select fct_order_details.order_id, fct_order_details.order_date,\n           fct_order_details.extended_price, fct_order_details.quantity,\n           fct_order_details.shipping_delay_days,\n           dim_categories.category_name, dim_products.product_name,\n           dim_customers.country, dim_employees.employee_full_name,\n           dim_suppliers.company_name as supplier_name\n    from northwind_warehouse.fct_order_details\n    join northwind_warehouse.dim_customers using (customer_pk)\n    join northwind_warehouse.dim_employees using (employee_pk)\n    join northwind_warehouse.dim_products using (product_pk)\n    join northwind_warehouse.dim_categories on dim_products.category_id = dim_categories.category_id\n    join northwind_warehouse.dim_suppliers on dim_products.supplier_id = dim_suppliers.supplier_id\n    where 1 = 1\n      [[and {{date_range}}]]\n      [[and {{category}}]]\n      [[and {{country}}]]\n      [[and {{employee}}]]\n      [[and {{product}}]]\n)\nselect product_name as product, sum(extended_price) as revenue from filtered group by 1 order by 2 desc limit 10;\n', 'queries/08_sales_by_country.sql': 'with filtered as (\n    select fct_order_details.order_id, fct_order_details.order_date,\n           fct_order_details.extended_price, fct_order_details.quantity,\n           fct_order_details.shipping_delay_days,\n           dim_categories.category_name, dim_products.product_name,\n           dim_customers.country, dim_employees.employee_full_name,\n           dim_suppliers.company_name as supplier_name\n    from northwind_warehouse.fct_order_details\n    join northwind_warehouse.dim_customers using (customer_pk)\n    join northwind_warehouse.dim_employees using (employee_pk)\n    join northwind_warehouse.dim_products using (product_pk)\n    join northwind_warehouse.dim_categories on dim_products.category_id = dim_categories.category_id\n    join northwind_warehouse.dim_suppliers on dim_products.supplier_id = dim_suppliers.supplier_id\n    where 1 = 1\n      [[and {{date_range}}]]\n      [[and {{category}}]]\n      [[and {{country}}]]\n      [[and {{employee}}]]\n      [[and {{product}}]]\n)\nselect country, sum(extended_price) as revenue from filtered group by 1 order by 2 desc;\n', 'queries/09_employee_performance.sql': 'with filtered as (\n    select fct_order_details.order_id, fct_order_details.order_date,\n           fct_order_details.extended_price, fct_order_details.quantity,\n           fct_order_details.shipping_delay_days,\n           dim_categories.category_name, dim_products.product_name,\n           dim_customers.country, dim_employees.employee_full_name,\n           dim_suppliers.company_name as supplier_name\n    from northwind_warehouse.fct_order_details\n    join northwind_warehouse.dim_customers using (customer_pk)\n    join northwind_warehouse.dim_employees using (employee_pk)\n    join northwind_warehouse.dim_products using (product_pk)\n    join northwind_warehouse.dim_categories on dim_products.category_id = dim_categories.category_id\n    join northwind_warehouse.dim_suppliers on dim_products.supplier_id = dim_suppliers.supplier_id\n    where 1 = 1\n      [[and {{date_range}}]]\n      [[and {{category}}]]\n      [[and {{country}}]]\n      [[and {{employee}}]]\n      [[and {{product}}]]\n)\nselect employee_full_name as employee, sum(extended_price) as revenue from filtered group by 1 order by 2 desc;\n', 'queries/10_shipping_performance.sql': 'with filtered as (\n    select fct_order_details.order_id, fct_order_details.order_date,\n           fct_order_details.extended_price, fct_order_details.quantity,\n           fct_order_details.shipping_delay_days,\n           dim_categories.category_name, dim_products.product_name,\n           dim_customers.country, dim_employees.employee_full_name,\n           dim_suppliers.company_name as supplier_name\n    from northwind_warehouse.fct_order_details\n    join northwind_warehouse.dim_customers using (customer_pk)\n    join northwind_warehouse.dim_employees using (employee_pk)\n    join northwind_warehouse.dim_products using (product_pk)\n    join northwind_warehouse.dim_categories on dim_products.category_id = dim_categories.category_id\n    join northwind_warehouse.dim_suppliers on dim_products.supplier_id = dim_suppliers.supplier_id\n    where 1 = 1\n      [[and {{date_range}}]]\n      [[and {{category}}]]\n      [[and {{country}}]]\n      [[and {{employee}}]]\n      [[and {{product}}]]\n)\nselect employee_full_name as employee, avg(shipping_delay_days) as average_shipping_days\nfrom (select distinct order_id, employee_full_name, shipping_delay_days from filtered) orders\nwhere shipping_delay_days is not null\ngroup by 1 order by 2 desc;\n'}
for name, content in support_files.items():
    path = Path(name)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(content, encoding="utf-8")


## 1. Imports and private connections

In [3]:
import json
from pathlib import Path
import re

import pandas as pd
import psycopg2
from connection import configure_connection
from create_dashboard import CARDS, FILTERS, create_dashboard

ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUT = ROOT / "results"
OUT.mkdir(exist_ok=True)
url = configure_connection()

## 2. Create the actual Metabase dashboard with every required chart and filter

In [4]:
client, state = create_dashboard()
dashboard_url = f"{client.url}/dashboard/{state['dashboard_id']}"

Created four KPIs, six charts and five connected filters.
Dashboard: http://localhost:8000/dashboard/2


## 3. Execute and save the ten real Metabase query results

In [5]:
results = {}
for filename, label, _, _, _ in CARDS:
    response = client.request("POST", f"/api/card/{state['card_ids'][filename]}/query", {"parameters": []})
    if response.get("status") != "completed":
        raise RuntimeError(f"Metabase query failed: {label}: {response.get('error')}")
    columns = [column["name"] for column in response["data"]["cols"]]
    frame = pd.DataFrame(response["data"]["rows"], columns=columns)
    frame.to_csv(OUT / f"{filename}.csv", index=False)
    results[filename] = response["data"]["rows"]
    print(label, "—", len(frame), "result rows")
    print(frame.head(5).to_string(index=False))

Total Revenue — 1 result rows
 total_revenue
  4.483866e+08
Total Orders — 1 result rows
 total_orders
        16282
Average Order Value — 1 result rows
 average_order_value
         27538.79334
Total Quantity Sold — 1 result rows
 total_quantity
       15538602
Sales Trend by Month — 136 result rows
               month    revenue
2012-07-01T00:00:00Z 2066219.40
2012-08-01T00:00:00Z 3556875.79
2012-09-01T00:00:00Z 3440144.98
2012-10-01T00:00:00Z 3201529.96
2012-11-01T00:00:00Z 2980494.74
Sales by Product Category — 8 result rows
      category      revenue
     Beverages 9.216318e+07
   Confections 6.633780e+07
  Meat/Poultry 6.488115e+07
Dairy Products 5.801812e+07
    Condiments 5.579513e+07
Top 10 Products by Revenue — 10 result rows
                product      revenue
          Côte de Blaye 53265895.235
Thüringer Rostbratwurst 24623469.232
        Mishi Kobe Niku 19423037.500
 Sir Rodney's Marmalade 16653807.360
       Carnarvon Tigers 12604671.875
Sales by Customer Country — 22

## 4. Verify all five filters and combined selection against PostgreSQL

In [6]:
# Use a real order as the test selection, so every filter selects actual data.
base_sql = (ROOT / "queries" / "01_total_revenue.sql").read_text(encoding="utf-8")
join_sql = base_sql.split("where 1 = 1")[0].replace("with filtered as (", "")
with psycopg2.connect(url) as connection:
    with connection.cursor() as cursor:
        cursor.execute(join_sql + " order by fct_order_details.order_id limit 1")
        sample = dict(zip([column.name for column in cursor.description], cursor.fetchone()))
        cursor.execute("select category_name from northwind_warehouse.dim_categories order by category_name")
        categories = [row[0] for row in cursor.fetchall()]
year = sample["order_date"].year
selections = {"date_range": f"{year}-01-01~{year}-12-31",
              "category": [sample["category_name"], next(c for c in categories if c != sample["category_name"])],
              "country": [sample["country"]], "employee": [sample["employee_full_name"]],
              "product": [sample["product_name"]]}
filter_checks = []
for name, selection in [(name, {name: value}) for name, value in selections.items()] + [("combined", selections)]:
    api_parameters = [{"id": key, "type": FILTERS[key][1], "target": ["dimension", ["template-tag", key]],
                       "value": value} for key, value in selection.items()]
    api_result = client.request("POST", f"/api/card/{state['card_ids']['01_total_revenue']}/query",
                                {"parameters": api_parameters})
    if api_result.get("status") != "completed":
        raise RuntimeError(f"Filter failed: {name}: {api_result.get('error')}")
    sql_values = []
    def replace_filter(match):
        key = match.group(1)
        if key not in selection:
            return ""
        _, _, table, field = FILTERS[key]
        if key == "date_range":
            start, end = selection[key].split("~")
            sql_values.extend([start, end])
            return f"and {table}.{field} between %s::date and %s::date"
        sql_values.append(selection[key])
        return f"and {table}.{field} = any(%s)"
    plain_sql = re.sub(r"\[\[and \{\{(\w+)\}\}\]\]", replace_filter, base_sql)
    with psycopg2.connect(url) as connection:
        with connection.cursor() as cursor:
            cursor.execute(plain_sql, sql_values)
            expected = float(cursor.fetchone()[0])
    actual = float(api_result["data"]["rows"][0][0])
    assert abs(actual - expected) < 0.01, f"Filter mismatch: {name}"
    filter_checks.append({"filter": name, "selection": selection, "revenue": actual, "verified": True})
    print("Verified filter:", name, "Revenue:", actual)

Verified filter: date_range Revenue: 40568672.36
Verified filter: category Revenue: 150181300.965
Verified filter: country Revenue: 53323305.7625
Verified filter: employee Revenue: 51386459.1025
Verified filter: product Revenue: 4234846.77
Verified filter: combined Revenue: 2667.0


## 5. Verify the layout and write the required dashboard explanation

In [7]:
dashboard = client.request("GET", f"/api/dashboard/{state['dashboard_id']}")
assert len(dashboard["dashcards"]) == 10 and len(dashboard["parameters"]) == 5
assert all(len(card["parameter_mappings"]) == 5 for card in dashboard["dashcards"])
verification = {"dashboard_url": dashboard_url, "cards": 10, "filters": 5,
               "filter_checks": filter_checks,
               "total_revenue": float(results["01_total_revenue"][0][0]),
               "total_orders": int(results["02_total_orders"][0][0]),
               "average_order_value": float(results["03_average_order_value"][0][0]),
               "total_quantity": int(results["04_total_quantity"][0][0])}
(OUT / "verification.json").write_text(json.dumps(verification, indent=2), encoding="utf-8")
report = f"""# TP7 — Northwind Order Details Dashboard

The actual Metabase dashboard is available locally at {dashboard_url} while
Metabase is running. Its analytical source is the TP6 Supabase
`northwind_warehouse` schema, not a copied local analytical dataset.
Its separate local PostgreSQL application database stores Metabase settings.
The repository includes the teacher's Docker Compose method. On this computer,
Docker was unavailable, so official Metabase v0.64.1 was launched with portable
Java 25 and the same PostgreSQL application-storage approach.

## Four KPIs
Total revenue: **{verification['total_revenue']:,.4f}**.
Unique orders: **{verification['total_orders']:,}**.
Average order value = revenue / unique orders: **{verification['average_order_value']:,.2f}**.
Total quantity: **{verification['total_quantity']:,}**.
The source does not establish the monetary currency; no currency symbol is invented.

## Six required charts
Monthly revenue trend; revenue by category; top 10 products by revenue;
revenue by customer country; revenue by employee; mean shipping days by employee.
Shipping averages use distinct orders, preventing orders with many detail lines
from receiving extra weight. Unshipped orders have NULL delay and are excluded
from the shipping mean, rather than treated as immediate shipments.

## Five interactive filters
Order date range, category, customer country, employee and product name.
Field filters support the date interval and multi-selection; every filter is
connected to all ten cards. Each filter and a combined selection were executed
through Metabase and reconciled with an independently parameterized PostgreSQL
query within one cent. The category check includes two simultaneous values.
All ten cards were queried successfully and their real results saved as CSVs.

Four KPIs occupy the top row; six charts follow in three pairs on one page.
Filters narrow the same order-detail set, so the average-order denominator counts
orders represented in the selection. Top products always means top 10 within
the chosen filters. The SQL joins the fact with all five specified dimensions.
No unrelated dashboard, extra visualization or public sharing link is created.

Assignment: https://github.com/ahmed3991/DataVisualisation-TP2025/blob/main/TP7/TP7.md
"""
(OUT / "REPORT.md").write_text(report, encoding="utf-8")
print("Dashboard and filter checks passed:", dashboard_url)

Dashboard and filter checks passed: http://localhost:8000/dashboard/2


## Required written explanation

In [8]:
from IPython.display import display, Markdown
display(Markdown((OUT / "REPORT.md").read_text(encoding="utf-8")))

# TP7 — Northwind Order Details Dashboard

The actual Metabase dashboard is available locally at http://localhost:8000/dashboard/2 while
Metabase is running. Its analytical source is the TP6 Supabase
`northwind_warehouse` schema, not a copied local analytical dataset.
Its separate local PostgreSQL application database stores Metabase settings.
The repository includes the teacher's Docker Compose method. On this computer,
Docker was unavailable, so official Metabase v0.64.1 was launched with portable
Java 25 and the same PostgreSQL application-storage approach.

## Four KPIs
Total revenue: **448,386,633.1695**.
Unique orders: **16,282**.
Average order value = revenue / unique orders: **27,538.79**.
Total quantity: **15,538,602**.
The source does not establish the monetary currency; no currency symbol is invented.

## Six required charts
Monthly revenue trend; revenue by category; top 10 products by revenue;
revenue by customer country; revenue by employee; mean shipping days by employee.
Shipping averages use distinct orders, preventing orders with many detail lines
from receiving extra weight. Unshipped orders have NULL delay and are excluded
from the shipping mean, rather than treated as immediate shipments.

## Five interactive filters
Order date range, category, customer country, employee and product name.
Field filters support the date interval and multi-selection; every filter is
connected to all ten cards. Each filter and a combined selection were executed
through Metabase and reconciled with an independently parameterized PostgreSQL
query within one cent. The category check includes two simultaneous values.
All ten cards were queried successfully and their real results saved as CSVs.

Four KPIs occupy the top row; six charts follow in three pairs on one page.
Filters narrow the same order-detail set, so the average-order denominator counts
orders represented in the selection. Top products always means top 10 within
the chosen filters. The SQL joins the fact with all five specified dimensions.
No unrelated dashboard, extra visualization or public sharing link is created.

Assignment: https://github.com/ahmed3991/DataVisualisation-TP2025/blob/main/TP7/TP7.md
